<a href="https://colab.research.google.com/github/mfletcher011/PYTHON-SCRIPTS-FOR-INVOICES/blob/main/Bulova_Citizen_Invoice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Step 1: Install necessary libraries
!pip install pdfplumber pandas openpyxl -q

# Step 2: Import libraries
import io
import re
import pandas as pd
import pdfplumber
from google.colab import files
import logging

# Configure logging
logging.basicConfig(level=logging.INFO)
logging.getLogger('pdfminer').setLevel(logging.WARNING)

def parse_citizen_invoice(file_content):
    """
    Parses the text from a Citizen Watch America invoice PDF.
    """
    items = []
    logging.info("Starting invoice parsing with updated script...")

    # Regex to find a line containing item details. It looks for a line that ends with a quantity and two dollar amounts.
    item_line_regex = re.compile(r'(\d+)\s+\$([\d,]+\.\d{2})\s+\$([\d,]+\.\d{2})\s*$')

    # --- THIS IS THE CORRECTED PART ---
    # More specific regex to find a style number. It must be 6-128 characters and contain at least one digit.
    # This prevents it from matching purely alphabetical words like "BULOVA".
    style_regex = re.compile(r'\b(?=\S*\d)\S{6,12}\b')

    with pdfplumber.open(io.BytesIO(file_content)) as pdf:
        all_lines = []
        for page in pdf.pages:
            all_lines.extend(page.extract_text(x_tolerance=2).split('\n'))

    current_style = ""
    current_description = ""

    for i, line in enumerate(all_lines):
        # Clean the line by removing the brand name before processing
        cleaned_line = line.replace("BULOVA", "").strip()

        price_match = item_line_regex.search(cleaned_line)

        if price_match:
            try:
                shipped_qty = int(price_match.group(1))
                unit_price = float(price_match.group(2).replace(",", ""))
                total_price = float(price_match.group(3).replace(",", ""))

                # The text before the prices is our area of interest for style and description
                interest_area = cleaned_line[:price_match.start()].strip()

                style_match = style_regex.search(interest_area)

                if style_match:
                    current_style = style_match.group(0) # Get the matched style
                    # Description is what remains after the style
                    current_description = interest_area[style_match.end():].strip()
                else:
                    # If style is not on the same line, check the cleaned line above
                    if i > 0:
                        prev_line_cleaned = all_lines[i-1].replace("BULOVA", "").strip()
                        prev_line_match = style_regex.search(prev_line_cleaned)
                        if prev_line_match:
                            current_style = prev_line_match.group(0)
                            current_description = interest_area # The whole chunk is the description

                if current_style:
                    # Final cleanup to remove any stray dots from the style
                    current_style = current_style.replace('.','').strip()

                    items.append({
                        "Item": current_style,
                        "Description": current_description,
                        "Shipped Qty": shipped_qty,
                        "Unit Price": unit_price,
                        "Total Price": total_price,
                    })
                    logging.info(f"  Parsed: Item={current_style}, Qty={shipped_qty}, Total={total_price}")
                    # Reset for the next item
                    current_style = ""
                    current_description = ""

            except (ValueError, IndexError) as e:
                logging.warning(f"Could not parse a potential item line: '{cleaned_line}'. Error: {e}")

    logging.info(f"--- Finished parsing. Found {len(items)} items. ---")
    return items


def main():
    """Main function to run the data extraction process."""
    print("Please upload the Citizen Invoice PDF file:")
    uploaded_files = files.upload()

    if not uploaded_files:
        print("No file uploaded. Exiting.")
        return

    file_name = list(uploaded_files.keys())[0]
    file_content = uploaded_files[file_name]

    invoice_number_match = re.search(r'Invoice Number:\s*(\d+)', file_content.decode('utf-8', errors='ignore'))
    invoice_number = invoice_number_match.group(1) if invoice_number_match else "Unknown"

    invoice_items = parse_citizen_invoice(file_content)

    if not invoice_items:
        print("\nNo items could be parsed from the invoice PDF. Please check the document format.")
        return

    df = pd.DataFrame(invoice_items)
    columns_ordered = ["Item", "Description", "Shipped Qty", "Unit Price", "Total Price"]
    df = df[columns_ordered]

    out_filename = f"Invoice_{invoice_number}_extracted.xlsx"
    df.to_excel(out_filename, index=False)

    print(f"\n✅ Success! Parsed {len(df)} items.")
    print(f"Downloading Excel file: '{out_filename}'")
    files.download(out_filename)

if __name__ == "__main__":
    main()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 124.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 126.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.
Please upload the Citizen Invoice PDF file:


Saving SID 80143537 – SO 113594 - COMMERCIAL INVOICE.pdf to SID 80143537 – SO 113594 - COMMERCIAL INVOICE.pdf

✅ Success! Parsed 37 items.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>